# The Catalyst Optimizer: what happens when you call an action

When you run an action like `df_5.show()`, Spark passes your code through the **Catalyst Optimizer**, Spark's "brain."

## The big idea

Remember **lazy evaluation**: transformations like `filter`, `groupBy` and `select` don't run anything. They just write down a plan.
When you call an **action** (`show()`, `count()`, `write`), Spark says "OK, now let's actually do it" and does two things:

1. **Optimize** the query: find the smartest way to do it
2. **Execute** the optimized plan: actually run it

### Analogy: a GPS 🗺️

You type: "Take me to Alexandria." The GPS:

- understands your request
- checks the place exists on its map
- thinks of smarter routes
- compares several routes (time, traffic)
- picks the best one
- then you drive

Spark does the same with your code. The rest of this notebook follows it step by step.

## Setup: build `df_5` again

This is the same query as in notebook 03: the top 3 (call type, zip code) pairs by number of calls.
Running this cell is fast, because it only uses transformations, so Spark just builds a plan.

In [ ]:
# Get a SparkSession (connected to Databricks through Databricks Connect)
from pyspark.shell import spark

# Read the source table: only the schema is loaded here, no data yet
fire_df = spark.read.table("dev.spark_db.sf_fire_calls")

# Transformations only: Spark writes down the plan but runs nothing
df_5 = (fire_df
        .select("callType", "Zipcode")             # keep 2 columns
        .where("callType is not null")             # drop rows with no call type
        .groupBy("callType", "Zipcode").count()    # count calls per pair
        .orderBy("count", ascending=False)         # most calls first
        .limit(3))                                 # top 3

## Step 0: the input

The input can be a **SQL query** or **DataFrame code**. Both go into the same pipeline and become the same plan.
That's why SQL and DataFrame code run equally fast in Spark.

## Step 1: Parsed Logical Plan 📝 ("understand the request")

Spark reads your code and writes down what you asked for, as a tree of steps (read it from the bottom up):

```
Limit 3
 └── Sort by count desc
      └── Aggregate: count, grouped by callType, Zipcode
           └── Filter: callType is not null
                └── Project: callType, Zipcode
                     └── Read: sf_fire_calls
```

At this point Spark **hasn't checked anything**. It doesn't know yet if `Zipcode` really exists or what type it is.
It's like writing down a customer's order before checking the kitchen has the ingredients.
This is also called the **Unresolved Logical Plan**.

## Step 2: Analysis → Analyzed Logical Plan 🔍 ("check it's valid")

Spark now asks the **Catalog** (the list of databases, tables and columns):

- Does the table exist? ✅
- Do the columns `callType` and `Zipcode` exist? ✅
- What are their types? (`string`, `int`, ...)

If something is wrong, **this is where the error appears**, before any data is read. For example, a typo in a column name:

```
AnalysisException: [UNRESOLVED_COLUMN] A column with name `Zipcod` cannot be resolved
```

### Try it: a typo is caught during analysis

In [ ]:
# A typo in the column name ("Zipcod" instead of "Zipcode")
bad_df = fire_df.select("callType", "Zipcod")

# Ask for the result. With Databricks Connect, the error usually shows up here,
# when the plan is sent to the server and analyzed.
try:
    bad_df.show()
except Exception as e:
    # Print only the first line of the error: it names the missing column
    print(type(e).__name__, "->", str(e).splitlines()[0])

## Step 3: Logical Optimization → Optimized Logical Plan ⚡ ("find a smarter way")

Spark applies rules that make the plan cheaper **without changing the result**. Common ones:

| Rule | What it does | Simple example |
|---|---|---|
| Predicate pushdown | Move filters as early as possible | Filter while reading, not after |
| Column pruning | Read only the columns that are needed | The table has 28 columns, the query needs 2 → read 2 |
| Constant folding | Pre-calculate fixed math | `price * (1 + 0.1)` → `price * 1.1` |
| Combine steps | Merge several filters or selects into one | Two `.filter()` calls → one |

**Analogy:** before going shopping, you reorder your list by aisle and cross off things you don't need. Same result, less walking.

This is why you can write simple, readable code: Spark rearranges it to be efficient for you.

## Step 4: Physical Planning → several Physical Plans 🛠️ ("how exactly to do it")

The logical plan says **what** to do. A physical plan says **how**, using real algorithms:

- how to read the files
- how to group (for example, a hash-based aggregation)
- how to join, if there are joins (broadcast join vs. sort-merge join, ...)
- where data must be **shuffled** (moved between machines)

Spark can create several possible physical plans for the same logical plan.

## Step 5: Cost Model → Selected Physical Plan 🏆 ("pick the best")

Spark estimates the cost of each physical plan (how much data, how much moving between machines) and picks the cheapest one.
Like the GPS comparing routes and choosing the fastest.

## Step 6: Code Generation → RDDs 🚀 ("actually run it")

Spark turns the chosen plan into fast compiled code. This feature is called **whole-stage code generation**, and it makes the job run almost like hand-written code.
The plan then runs as **RDD** operations, Spark's low-level engine (Spark Core).
The work is split into **tasks**, sent to the machines, and the results come back.

### Run the action

Only now does the whole pipeline above actually run.

In [ ]:
# Action: runs steps 1 to 6 and prints the result table
df_5.show()

## The whole journey in one picture

```
df_5.show()  (action!)
   │
1. Parsed Logical Plan      → "what did the user ask?"              (not checked yet)
   │
2. Analyzed Logical Plan    → check with Catalog: tables, columns, types   (errors appear here)
   │
3. Optimized Logical Plan   → smarter order: filter early, read fewer columns
   │
4. Physical Plans           → several ways to actually execute
   │
5. Selected Physical Plan   → cost model picks the cheapest
   │
6. Code Generation → RDDs   → fast code runs on the machines → results
```

## Can we see these plans?

Yes: `explain(True)` prints them. It does **not** run a job. The sections match the steps above:

```
== Parsed Logical Plan ==       ← Step 1
== Analyzed Logical Plan ==     ← Step 2
== Optimized Logical Plan ==    ← Step 3
== Physical Plan ==             ← Step 5 (the selected one)
```

What to look for in the output:

- **Analyzed plan:** columns now have types and IDs, for example `callType#12: string`
- **Optimized plan:** only `callType` and `Zipcode` are read from the table (column pruning)
- **Physical plan:** `HashAggregate` (the grouping algorithm) and `Exchange` (a shuffle between machines)

In [ ]:
# Print all 4 plans (parsed, analyzed, optimized, physical). No job is run.
df_5.explain(True)

### Other `explain` modes

`explain()` takes a `mode` argument:

| Mode | Shows |
|---|---|
| `"simple"` (default) | Only the physical plan |
| `"extended"` | All 4 plans, same as `explain(True)` |
| `"formatted"` | The physical plan as a short outline plus details for each step, easier to read |
| `"cost"` | The optimized plan with size estimates, when statistics are available |
| `"codegen"` | The generated code, if any |

In [ ]:
# Easier-to-read physical plan: an outline first, then details for each numbered step
df_5.explain(mode="formatted")

## Key takeaways

- Transformations only build a plan; an action starts the Catalyst pipeline.
- Errors like a wrong column name are caught in **analysis**, before any data is read.
- The **optimizer** rewrites your plan (filter early, read fewer columns), so readable code is also fast code.
- SQL and DataFrame code become the same plan, so neither is faster.
- Use `explain(True)` to see the plans Spark built.